In [8]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

In [ ]:
factor = pd.read_parquet('C:/Users/User/OneDrive - CUHK-Shenzhen/data/factor/sample_oot.parquet')
ind = pd.read_csv('C:/Users/User/OneDrive - CUHK-Shenzhen/data/industry.csv')
factor = factor.merge(ind[['ts_code', 'l1_name']], on='ts_code', how='left')
factor = pd.get_dummies(factor, columns=['l1_name'], drop_first=True)
#daily_corr = factor.groupby('trade_date').apply(lambda x: x[factor.columns[8:]].corrwith(x['Y_20d'])).reset_index()

In [ ]:
# 2. 行业+市值中性化
def neutralize_date(group):
    X = sm.add_constant(group[group.columns[-30:].tolist()+['lncap']])
    model = sm.OLS(group['l1_name'], X, missing='drop')
    res = model.fit()
    group['factor_neutral'] = res.resid
    return group

factor = factor.groupby('trade_date').apply(neutralize_date).reset_index(drop=True)